## 1. Setup

In [ ]:
import os, re, json, math, time, heapq, collections
from pathlib import Path

import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)


# COMPETITION_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
CORPUS_DIR = "./RAG/wiki_corpus"
OUTPUT_DIR = "/"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)


device: cpu


## 2. Data

In [ ]:
# ---- data settings ----
MAX_CHARS = None      # None = whole corpus; set e.g. 2_000_000 for a fast trial run
MIN_DOC_CHARS = 500   # skip near-empty stubs


def clean_md(raw: str) -> str:
    lines = []
    for ln in raw.split('\n'):
        s = ln.strip()
        if s.startswith('>') or s == '---':      # metadata header, separator rule
            continue
        lines.append(ln)
    text = '\n'.join(lines)

    # inline maths: drop $...$ spans that contain a backslash
    text = re.sub(r'\$[^$]{0,2000}?\$',
                  lambda m: ' ' if '\\' in m.group(0) else m.group(0), text, flags=re.S)

    text = re.sub(r'[ \t]+', ' ', text)              # collapse repeated spaces
    text = re.sub(r' +([,.;:!?%)\]])', r'\1', text)  # "System , centered" -> "System, centered"
    text = re.sub(r'([(\[]) +', r'\1', text)         # "( 1905 )" -> "(1905)"
    text = re.sub(r'(\w) +-(\w)', r'\1-\2', text)    # "torus -shaped" -> "torus-shaped"
    text = re.sub(r'\n{3,}', '\n\n', text)           # collapse blank-line runs
    return text.strip()


files = sorted(Path(CORPUS_DIR).glob('*.md'))        # ignores dataset-metadata.json
raw_chars, docs = 0, []
for f in files:
    raw = f.read_text(encoding='utf-8', errors='ignore')
    raw_chars += len(raw)
    doc = clean_md(raw)
    if len(doc) > MIN_DOC_CHARS:
        docs.append(doc)

text = '\n\n'.join(docs)
print(f'{len(files)} files -> {len(docs)} kept')
print(f'{raw_chars:,} raw chars -> {len(text):,} cleaned ({100*(1-len(text)/raw_chars):.1f}% removed)')
if MAX_CHARS:
    text = text[:MAX_CHARS]
    print(f'truncated to {len(text):,}')
print(f'{len(set(text)):,} unique characters | leftover " ," artifacts:',
      len(re.findall(r' [,.]', text)))
print('-' * 70)
print(text[:500])

## 3. Tokenizers — two of them

### 3a. Character-level

In [ ]:
class CharTokenizer:
    "One token per character. Rare characters collapse to <unk>."

    def __init__(self, itos):
        self.itos = list(itos)
        self.stoi = {c: i for i, c in enumerate(self.itos)}
        self.unk = self.stoi['<unk>']

    @classmethod
    def from_text(cls, text, min_freq=100):
        # the corpus has ~780 distinct characters, mostly one-off unicode junk;
        # keeping only the common ones stops the embedding table filling with noise
        counts = collections.Counter(text)
        chars = sorted([c for c, n in counts.items() if n >= min_freq])
        print(f'kept {len(chars)} of {len(counts)} characters (min_freq={min_freq})')
        return cls(chars + ['<unk>'])

    @property
    def vocab_size(self):
        return len(self.itos)

    def encode(self, s):
        return [self.stoi.get(c, self.unk) for c in s]

    def decode(self, ids):
        return ''.join(self.itos[i] for i in ids)

    def save(self, path):
        json.dump({'type': 'char', 'itos': self.itos}, open(path, 'w', encoding='utf-8'))

    @classmethod
    def load(cls, path):
        return cls(json.load(open(path, encoding='utf-8'))['itos'])

### 3b. Byte-pair encoding

In [ ]:
# leading-space-attached chunks; whitespace runs are their own chunk
PAT = re.compile(r"'(?:s|t|re|ve|m|ll|d)| ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+")


def train_bpe(text, vocab_size, verbose=True):
    "Returns (chars, merges); merges is the ordered list of (a, b) pairs that were joined."
    freqs = collections.Counter(PAT.findall(text))
    words = [list(w) for w in freqs]           # each word as a list of symbols
    counts = list(freqs.values())              # how often that word occurs
    if verbose:
        print(f'{sum(counts):,} chunks, {len(words):,} unique')

    pair_counts = collections.Counter()        # pair -> total count
    pair_where = collections.defaultdict(set)  # pair -> which words contain it
    for i, w in enumerate(words):
        for p in zip(w, w[1:]):
            pair_counts[p] += counts[i]
            pair_where[p].add(i)

    heap = [(-c, p) for p, c in pair_counts.items()]
    heapq.heapify(heap)

    chars = sorted({c for w in words for c in w})
    merges = []
    target = vocab_size - len(chars) - 1       # -1 leaves room for <unk>
    t0 = time.time()

    while len(merges) < target and heap:
        negc, pair = heapq.heappop(heap)
        if pair_counts.get(pair, 0) != -negc:  # stale heap entry
            continue
        if -negc < 2:                          # nothing frequent left to merge
            break
        new_sym = pair[0] + pair[1]
        merges.append(pair)
        touched = set()

        for i in list(pair_where[pair]):
            w, f = words[i], counts[i]
            for p in zip(w, w[1:]):            # withdraw this word's old pairs
                pair_counts[p] -= f
                touched.add(p)
            new_w, j = [], 0
            while j < len(w):                  # apply the merge left to right
                if j < len(w) - 1 and w[j] == pair[0] and w[j + 1] == pair[1]:
                    new_w.append(new_sym); j += 2
                else:
                    new_w.append(w[j]); j += 1
            words[i] = new_w
            for p in zip(new_w, new_w[1:]):    # and put the new pairs back
                pair_counts[p] += f
                pair_where[p].add(i)
                touched.add(p)

        pair_counts.pop(pair, None)
        pair_where.pop(pair, None)
        for p in touched:                      # re-publish changed counts
            c = pair_counts.get(p, 0)
            if c > 0:
                heapq.heappush(heap, (-c, p))

        if verbose and len(merges) % 1000 == 0:
            print(f'  merge {len(merges):5d}/{target}  {new_sym!r}  ({time.time()-t0:.0f}s)')

    if verbose:
        print(f'{len(chars)} chars + {len(merges)} merges in {time.time()-t0:.0f}s')
    return chars, merges

In [ ]:
class BPETokenizer:
    "Characters plus learned merges."

    def __init__(self, chars, merges):
        self.merges = [tuple(m) for m in merges]
        self.ranks = {p: i for i, p in enumerate(self.merges)}
        self.itos = list(chars) + [a + b for a, b in self.merges] + ['<unk>']
        self.stoi = {s: i for i, s in enumerate(self.itos)}
        self.unk = len(self.itos) - 1
        self._cache = {}

    @classmethod
    def from_text(cls, text, vocab_size=4000):
        return cls(*train_bpe(text, vocab_size))

    @property
    def vocab_size(self):
        return len(self.itos)

    def _bpe(self, chunk):
        "Apply merges to one chunk, always taking the lowest-ranked (earliest) merge first."
        if chunk in self._cache:
            return self._cache[chunk]
        w = list(chunk)
        while len(w) > 1:
            best_rank, best_i = None, None
            for i in range(len(w) - 1):
                r = self.ranks.get((w[i], w[i + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_i = r, i
            if best_i is None:
                break
            w[best_i:best_i + 2] = [w[best_i] + w[best_i + 1]]
        self._cache[chunk] = w
        return w

    def encode(self, s):
        return [self.stoi.get(sym, self.unk)
                for chunk in PAT.findall(s) for sym in self._bpe(chunk)]

    def decode(self, ids):
        return ''.join(self.itos[i] for i in ids)

    def save(self, path):
        json.dump({'type': 'bpe', 'itos': self.itos, 'merges': self.merges},
                  open(path, 'w', encoding='utf-8'))

    @classmethod
    def load(cls, path):
        d = json.load(open(path, encoding='utf-8'))
        chars = d['itos'][:len(d['itos']) - len(d['merges']) - 1]
        return cls(chars, d['merges'])

### 3c. Build both and compare

In [ ]:
# ---- tokenizer settings ----
CHAR_MIN_FREQ = 100    # characters rarer than this become <unk>
BPE_VOCAB = 4000       # total BPE vocabulary (characters + merges + <unk>)

tok_char = CharTokenizer.from_text(text, min_freq=CHAR_MIN_FREQ)
tok_bpe = BPETokenizer.from_text(text, vocab_size=BPE_VOCAB)

sample = 'Einstein published the theory of general relativity in 1915.'
probe = text[:200_000]                    # measure the ratio on a slice, it is enough

print(f'\n{"":16s} {"vocab":>7s} {"chars/token":>12s}')
for name, t in [('character', tok_char), ('BPE', tok_bpe)]:
    ratio = len(probe) / len(t.encode(probe))
    print(f'{name:16s} {t.vocab_size:7d} {ratio:12.2f}')
    assert t.decode(t.encode(sample)) == sample, f'{name}: round-trip failed'

print('\nround-trip OK for both')
print('\nchar pieces:', [tok_char.itos[i] for i in tok_char.encode(sample)][:18], '...')
print('BPE  pieces:', [tok_bpe.itos[i] for i in tok_bpe.encode(sample)])

## 4. Picking one and encoding the corpus

In [ ]:
# ---- what to train with ----
TOKENIZER = 'bpe'      # 'bpe' or 'char'
VAL_FRAC = 0.1         # last 10% of the corpus is held out

tok = tok_bpe if TOKENIZER == 'bpe' else tok_char
print('using:', TOKENIZER, '| vocab', tok.vocab_size)

t0 = time.time()
data = torch.tensor(tok.encode(text), dtype=torch.long)
chars_per_token = len(text) / len(data)      # needed for bits-per-character later
print(f'{len(text):,} chars -> {len(data):,} tokens in {time.time()-t0:.0f}s '
      f'({chars_per_token:.2f} chars/token)')

n = int((1 - VAL_FRAC) * len(data))
train_data, val_data = data[:n], data[n:]
print(f'train {len(train_data):,} | val {len(val_data):,}')

## 5. The model

Layer by layer:

- **`Head`** — one self-attention head. Queries, keys and values are linear projections;
  `q @ k.T / sqrt(d)` scores every pair of positions; the lower-triangular mask makes it
  *causal* so position `t` only sees positions `<= t`; softmax turns scores into weights; the
  output is the weighted sum of values.
- **`MultiHeadAttention`** — several heads in parallel, concatenated and projected back.
- **`FeedForward`** — the per-position MLP, 4x hidden expansion.
- **`Block`** — `x = x + attn(ln1(x))` then `x = x + ffwd(ln2(x))`: pre-norm residuals.
- **`GPTLanguageModel`** — token embedding + learned position embedding, N blocks, a final
  LayerNorm, and a linear head producing one logit per vocabulary entry.

In [ ]:
# ---- model settings ----
block_size = 128    # context length in tokens
n_embd = 256        # embedding dimension
n_head = 4          # attention heads, so head_size = 256 // 4 = 64
n_layer = 4         # transformer blocks
dropout = 0.1

if device == 'cpu':                       # keep a CPU run merely slow, not hopeless
    block_size, n_embd, n_layer = 64, 128, 3

# with the character tokenizer each token is ~3.5x less text, so reach further back
if TOKENIZER == 'char':
    block_size *= 2

print(f'block_size {block_size} | n_embd {n_embd} | n_head {n_head} | n_layer {n_layer}')

In [ ]:
class Head(nn.Module):
    "One head of causal self-attention."

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k, q = self.key(x), self.query(x)                               # (B,T,hs)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5             # (B,T,T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))    # causal mask
        wei = self.dropout(F.softmax(wei, dim=-1))
        return wei @ self.value(x)                                      # (B,T,hs)


class MultiHeadAttention(nn.Module):
    "Several attention heads running in parallel."

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.proj(torch.cat([h(x) for h in self.heads], dim=-1)))


class FeedForward(nn.Module):
    "Position-wise MLP."

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    "Communication (attention) followed by computation (MLP)."

    def __init__(self, n_embd, n_head):
        super().__init__()
        self.sa = MultiHeadAttention(n_head, n_embd // n_head)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))       # residual around attention
        x = x + self.ffwd(self.ln2(x))     # residual around the MLP
        return x

In [ ]:
class GPTLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)                                    # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))  # (T,C)
        x = self.ln_f(self.blocks(tok_emb + pos_emb))
        logits = self.lm_head(x)                                                     # (B,T,V)

        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, banned=None):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])   # never feed more than the context
            logits = logits[:, -1, :] / temperature  # only the last position predicts next
            if banned is not None:                   # keep <unk> out of the output
                logits[:, banned] = -float('inf')
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat((idx, torch.multinomial(probs, num_samples=1)), dim=1)
        return idx


model = GPTLanguageModel(tok.vocab_size).to(device)
print(f'{sum(p.numel() for p in model.parameters())/1e6:.2f} M parameters')

## 6. Training

`get_batch` picks random windows out of the token stream; the targets are the same window
shifted one step, so every position predicts the next token.

In [ ]:
# ---- training settings ----
batch_size = 64
max_iters = 4000
eval_interval = 250
eval_iters = 50
learning_rate = 3e-4

if device == 'cpu':
    batch_size, max_iters, eval_interval, eval_iters = 16, 500, 100, 20


def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])   # inputs shifted by one
    return x.to(device), y.to(device)


@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


xb, yb = get_batch('train')
print('batch:', xb.shape, '| loss at init:', estimate_loss(),
      f'(uniform guess would be {math.log(tok.vocab_size):.2f})')

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
history = []
t0 = time.time()

for it in range(max_iters + 1):
    if it % eval_interval == 0 or it == max_iters:
        losses = estimate_loss()
        history.append((it, losses['train'], losses['val']))
        print(f"step {it:5d} | train {losses['train']:.4f} | val {losses['val']:.4f} "
              f"| {time.time()-t0:.0f}s")

    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f'trained in {(time.time()-t0)/60:.1f} min')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot([h[0] for h in history], [h[1] for h in history], label='train')
plt.plot([h[0] for h in history], [h[2] for h in history], label='val')
plt.xlabel('iteration'); plt.ylabel('cross-entropy loss')
plt.title(f'Training curve ({TOKENIZER} tokenizer)'); plt.legend(); plt.grid(alpha=.3)
plt.show()

val_loss = history[-1][2]
bpc = val_loss / math.log(2) / chars_per_token
print(f'val loss {val_loss:.4f} | perplexity {math.exp(val_loss):.1f} per token')
print(f'bits per character {bpc:.3f}   <-- compare the two tokenizers on THIS number')

## 7. Generating text

In [ ]:
def generate(prompt='', max_new_tokens=300, temperature=0.8, top_k=50):
    model.eval()
    ids = tok.encode(prompt) if prompt else [tok.stoi.get('\n', 0)]
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    out = model.generate(idx, max_new_tokens, temperature=temperature,
                         top_k=top_k, banned=tok.unk)
    return tok.decode(out[0].tolist())


for p in ['The theory of relativity', 'Albert Einstein was', 'In quantum mechanics,']:
    print('=' * 70)
    print(generate(p, max_new_tokens=180))

In [ ]:
# temperature controls how adventurous the sampling is
for t in [0.5, 0.8, 1.2]:
    print('=' * 70, f'\n[temperature = {t}]')
    print(generate('Energy is', max_new_tokens=100, temperature=t))

## 8. Save

In [ ]:
cfg = dict(tokenizer=TOKENIZER, vocab_size=tok.vocab_size, block_size=block_size,
           n_embd=n_embd, n_head=n_head, n_layer=n_layer, dropout=dropout)

torch.save({'model': model.state_dict(), 'config': cfg}, f'{OUT_DIR}/gpt_wiki.pt')
tok.save(f'{OUT_DIR}/tokenizer_{TOKENIZER}.json')
print('saved', cfg)

# to reload:
# tok2 = BPETokenizer.load(f'{OUT_DIR}/tokenizer_bpe.json')     # or CharTokenizer.load(...)
# ck = torch.load(f'{OUT_DIR}/gpt_wiki.pt', map_location=device)
# m2 = GPTLanguageModel(ck['config']['vocab_size']).to(device); m2.load_state_dict(ck['model'])

---

# Part 2 — Using the pretrained model on `train.csv`

## 9. The dataset and the split

In [ ]:
import numpy as np
import pandas as pd

# ---- split settings ----
VAL_PROMPT_FRAC = 0.1
SPLIT_SEED = 1337

TRAIN_CSV = next((p for p in ['train.csv', '../train.csv', '/kaggle/input/train.csv']
                  if os.path.exists(p)), None)
assert TRAIN_CSV, 'train.csv not found'

OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
df = pd.read_csv(TRAIN_CSV)
df['label'] = df['answer'].map({c: i for i, c in enumerate(OPTION_COLS)})
print('raw:', df.shape, '| answers:', df['answer'].value_counts().to_dict())

before = len(df)
df = df.drop_duplicates(subset=['prompt'] + OPTION_COLS).reset_index(drop=True)
print(f'dropped {before - len(df)} duplicate rows -> {len(df)}')

rng = np.random.RandomState(SPLIT_SEED)
prompts = df['prompt'].unique()
rng.shuffle(prompts)
val_prompts = set(prompts[:int(VAL_PROMPT_FRAC * len(prompts))])
is_val = df['prompt'].isin(val_prompts)

val_df = df[is_val].reset_index(drop=True)
trn_df = df[~is_val].reset_index(drop=True)
y_val = val_df['label'].values
assert not (set(trn_df['prompt']) & set(val_df['prompt'])), 'a prompt leaked across the split'
print(f'train {len(trn_df)} | val {len(val_df)} | no shared prompts')

r = trn_df.iloc[0]
print('\nQ:', r['prompt'][:160])
for c in OPTION_COLS:
    print(f'  {c}: {str(r[c])[:100]}')
print('answer:', r['answer'])

### Metrics

In [ ]:
def map_at_3(scores, labels):
    "Mean average precision @ 3 over the 5 OPTION_COLS."
    order = np.argsort(-np.asarray(scores), axis=1)[:, :3]
    total = 0.0
    for row, y in zip(order, labels):
        for k, p in enumerate(row):
            if p == y:
                total += 1.0 / (k + 1)
                break
    return total / len(labels)


def metrics(scores, labels=None):
    "accuracy, top-2, MAP@3 and macro precision/recall/F1 over the five letters."
    labels = y_val if labels is None else np.asarray(labels)
    scores = np.asarray(scores)
    pred = scores.argmax(1)
    top2 = np.argsort(-scores, axis=1)[:, :2]

    precs, recs, f1s = [], [], []
    for c in range(len(OPTION_COLS)):
        tp = int(((pred == c) & (labels == c)).sum())
        p = tp / max(int((pred == c).sum()), 1)      # of what we called c, how much was c
        r = tp / max(int((labels == c).sum()), 1)    # of the real c, how much we found
        precs.append(p); recs.append(r)
        f1s.append(2 * p * r / (p + r) if p + r else 0.0)

    return {
        'acc': (pred == labels).mean(),
        'top2': np.mean([labels[i] in top2[i] for i in range(len(labels))]),
        'prec': np.mean(precs), 'rec': np.mean(recs), 'f1': np.mean(f1s),
        'map3': map_at_3(scores, labels),
    }


HEADER = (f'{"":26s} {"acc":>6s} {"top2":>6s} {"prec":>6s} {"rec":>6s} '
          f'{"f1":>6s} {"MAP@3":>6s}     (prec/rec/f1 are macro)')


def report(name, scores, labels=None):
    m = metrics(scores, labels)
    print(f'{name:26s} {m["acc"]:6.4f} {m["top2"]:6.4f} {m["prec"]:6.4f} '
          f'{m["rec"]:6.4f} {m["f1"]:6.4f} {m["map3"]:6.4f}')
    return m


def per_class_table(scores, labels=None):
    "Precision / recall / F1 / support for each answer letter."
    labels = y_val if labels is None else np.asarray(labels)
    pred = np.asarray(scores).argmax(1)
    rows = []
    for c, letter in enumerate(OPTION_COLS):
        tp = int(((pred == c) & (labels == c)).sum())
        p = tp / max(int((pred == c).sum()), 1)
        r = tp / max(int((labels == c).sum()), 1)
        rows.append({'letter': letter, 'precision': round(p, 4), 'recall': round(r, 4),
                     'f1': round(2 * p * r / (p + r) if p + r else 0.0, 4),
                     'support': int((labels == c).sum()), 'predicted': int((pred == c).sum())})
    return pd.DataFrame(rows).set_index('letter')


rand = rng.rand(len(val_df), 5)
maj = np.zeros((len(val_df), 5)); maj[:, 1] = 1                    # always answer B
lengths = np.stack([val_df[c].astype(str).str.len().values for c in OPTION_COLS], 1)

print(HEADER)
report('random guessing', rand)
report('always B', maj)
report('longest option', lengths)     # a baseline with no language understanding at all

## 10. Zero-shot: score OPTION_COLS by language-model likelihood

In [ ]:
def encode_pair(question, option, max_len=None):
    "Returns (token ids, how many of them belong to the option)."
    max_len = max_len or block_size
    q = tok.encode(str(question).strip()) if question else []
    o = tok.encode('\n' + str(option).strip())
    ids = q + o
    if len(ids) > max_len:
        ids = ids[-max_len:]                  # keep the tail so the option survives
    return ids, min(len(o), len(ids) - 1)


def encode_df(frame, use_question=True):
    "use_question=False is the control in section 12: option text on its own."
    return [[encode_pair(r['prompt'] if use_question else '', r[c]) for c in OPTION_COLS]
            for _, r in frame.iterrows()]


trn_enc, val_enc = encode_df(trn_df), encode_df(val_df)
lens = [len(ids) for row in trn_enc for ids, _ in row]
print(f'sequence length: mean {np.mean(lens):.0f}, max {max(lens)} (capped at {block_size})')

In [ ]:
@torch.no_grad()
def lm_scores(encoded, batch=40):
    "Mean log-prob of the option tokens for every (question, option) pair."
    model.eval()
    flat = [p for row in encoded for p in row]
    out = []
    for s in range(0, len(flat), batch):
        chunk = flat[s:s + batch]
        T = max(len(ids) for ids, _ in chunk)
        x = torch.zeros(len(chunk), T, dtype=torch.long, device=device)
        for i, (ids, _) in enumerate(chunk):
            x[i, :len(ids)] = torch.tensor(ids, device=device)
        logits, _ = model(x)
        logp = F.log_softmax(logits, dim=-1)
        tokl = logp[:, :-1].gather(2, x[:, 1:, None]).squeeze(2)   # log-prob of the real next token
        for i, (ids, n_opt) in enumerate(chunk):
            out.append(tokl[i, :len(ids) - 1][-n_opt:].mean().item())
    return np.array(out).reshape(len(encoded), 5)


val_zero = lm_scores(val_enc)
print(HEADER)
report('zero-shot LM likelihood', val_zero)

## 11. Fine-tuning a scoring head

In [ ]:
class GPTMultipleChoice(nn.Module):
    "Pretrained GPT trunk + a linear scorer over the final hidden state."

    def __init__(self, gpt):
        super().__init__()
        self.gpt = gpt
        self.score = nn.Linear(n_embd, 1)

    def forward(self, x, last, targets=None):
        BN, T = x.shape                                   # BN = batch * 5 OPTION_COLS
        tok_emb = self.gpt.token_embedding_table(x)
        pos_emb = self.gpt.position_embedding_table(torch.arange(T, device=x.device))
        h = self.gpt.ln_f(self.gpt.blocks(tok_emb + pos_emb))
        h = h[torch.arange(BN, device=x.device), last]    # (BN, C) at the last real token
        logits = self.score(h).view(-1, 5)                # (batch, 5)
        loss = F.cross_entropy(logits, targets) if targets is not None else None
        return logits, loss


def mc_batch(encoded, labels, idx):
    seqs = [ids for i in idx for ids, _ in encoded[i]]
    T = max(len(s) for s in seqs)
    x = torch.zeros(len(seqs), T, dtype=torch.long)
    for i, s in enumerate(seqs):
        x[i, :len(s)] = torch.tensor(s)
    last = torch.tensor([len(s) - 1 for s in seqs])
    y = torch.tensor([labels[i] for i in idx])
    return x.to(device), last.to(device), y.to(device)


@torch.no_grad()
def mc_predict(mc, encoded, bs=8):
    mc.eval()
    dummy = [0] * len(encoded)
    out = []
    for s in range(0, len(encoded), bs):
        x, last, _ = mc_batch(encoded, dummy, range(s, min(s + bs, len(encoded))))
        out.append(mc(x, last)[0].float().cpu().numpy())
    return np.concatenate(out)

In [ ]:
# ---- fine-tuning settings ----
MC_EPOCHS = 3
MC_BATCH = 8           # questions per step, so 5x that many sequences
MC_LR = 3e-4


def finetune(gpt, trn_e=None, val_e=None, epochs=MC_EPOCHS, tag='fine-tuned'):
    trn_e = trn_enc if trn_e is None else trn_e
    val_e = val_enc if val_e is None else val_e
    mc = GPTMultipleChoice(gpt).to(device)
    opt = torch.optim.AdamW(mc.parameters(), lr=MC_LR)
    y_trn = trn_df['label'].values
    hist = []

    for ep in range(1, epochs + 1):
        mc.train()
        order = rng.permutation(len(trn_e))
        running, nb, t0 = 0.0, 0, time.time()
        for s in range(0, len(order) - MC_BATCH + 1, MC_BATCH):
            x, last, y = mc_batch(trn_e, y_trn, order[s:s + MC_BATCH])
            _, loss = mc(x, last, y)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            running += loss.item(); nb += 1

        m = metrics(mc_predict(mc, val_e))
        hist.append((ep, running / nb, m['acc'], m['map3']))
        print(f'[{tag}] epoch {ep} | train loss {running/nb:.4f} | val acc {m["acc"]:.4f} '
              f'| macro f1 {m["f1"]:.4f} | MAP@3 {m["map3"]:.4f} | {time.time()-t0:.0f}s')
    return mc, hist


mc_model, mc_hist = finetune(model)

In [ ]:
# control 1: the same fine-tune from a randomly initialised trunk.
# whatever gap opens up is what pretraining on the wiki corpus bought us.
torch.manual_seed(1337)
scratch_gpt = GPTLanguageModel(tok.vocab_size).to(device)
mc_scratch, sc_hist = finetune(scratch_gpt, tag='random trunk')

## 12. Control: is the model reading the question at all?

In [ ]:
trn_noq, val_noq = encode_df(trn_df, use_question=False), encode_df(val_df, use_question=False)

torch.manual_seed(1337)
noq_gpt = GPTLanguageModel(tok.vocab_size).to(device)
mc_noq, noq_hist = finetune(noq_gpt, trn_noq, val_noq, tag='OPTION_COLS only')

In [ ]:
val_ft = mc_predict(mc_model, val_enc)
val_sc = mc_predict(mc_scratch, val_enc)
val_nq = mc_predict(mc_noq, val_noq)

print('VALIDATION RESULTS (deduplicated, prompt-grouped split)')
print(HEADER)
print('-' * 76)
report('random guessing', rand)
report('always B', maj)
report('longest option', lengths)
print('-' * 76)
report('zero-shot LM likelihood', val_zero)
report('fine-tuned, random trunk', val_sc)
m_ft = report('fine-tuned, pretrained', val_ft)
print('-' * 76)
m_nq = report('CONTROL: OPTION_COLS only', val_nq)

print(f'\nreading the question is worth {m_ft["acc"] - m_nq["acc"]:+.4f} accuracy '
      f'and {m_ft["map3"] - m_nq["map3"]:+.4f} MAP@3 over not reading it')
print('a gap near zero means the score is an artifact of the OPTION_COLS, not comprehension')

print('\nper-letter breakdown, fine-tuned model:')
print(per_class_table(val_ft))

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot([h[0] for h in mc_hist], [h[3] for h in mc_hist], 'o-', label='pretrained trunk')
plt.plot([h[0] for h in sc_hist], [h[3] for h in sc_hist], 's-', label='random trunk')
plt.plot([h[0] for h in noq_hist], [h[3] for h in noq_hist], '^--', c='crimson',
         label='CONTROL: OPTION_COLS only')
plt.axhline(0.3667, ls='--', c='grey', label='random guessing')
plt.xlabel('epoch'); plt.ylabel('val MAP@3'); plt.legend(); plt.grid(alpha=.3)
plt.title('Fine-tuning on train.csv'); plt.show()

pred = val_ft.argmax(1)
print(pd.crosstab(pd.Series([OPTION_COLS[i] for i in y_val], name='true'),
                  pd.Series([OPTION_COLS[i] for i in pred], name='pred')))

## 13. Predictions on `test.csv`

In [ ]:
TEST_CSV = next((p for p in ['test.csv', '../test.csv', '/kaggle/input/test.csv']
                 if os.path.exists(p)), None)

if TEST_CSV:
    test_df = pd.read_csv(TEST_CSV)
    test_logits = mc_predict(mc_model, encode_df(test_df))
    top3 = np.argsort(-test_logits, axis=1)[:, :3]

    sub = pd.DataFrame({'ID': test_df['id'],
                        'Prediction': [' '.join(OPTION_COLS[j] for j in row) for row in top3]})
    sub.to_csv(f'{OUT_DIR}/submission.csv', index=False)
    print(sub.head(), '\nwrote submission.csv', sub.shape)
else:
    print('test.csv not found, skipping')

torch.save({'model': mc_model.state_dict(), 'config': cfg}, f'{OUT_DIR}/gpt_wiki_mc.pt')
print('saved fine-tuned classifier')